<div style="background:linear-gradient(135deg, rgba(249,112,102,0.18), rgba(249,112,102,0.02)); border-left:6px solid #F97066; border-radius:10px; padding:20px 24px; margin-bottom:20px;">
<h1 style="margin:0; color:#F97066; font-size:1.8em;">🧰 Múltiples Herramientas</h1>
<p style="margin:6px 0 0; opacity:0.8;">Unidad 5 — Cómo un agente elige, entre varias herramientas, cuál (o ninguna) usar</p>
</div>

Un agente útil casi nunca tiene una sola herramienta disponible. Este notebook construye un agente con tres herramientas distintas y muy diferentes entre sí, y observa cómo el LLM decide cuál invocar según la pregunta — y, tan importante como eso, cuándo decide no invocar ninguna.

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0 24px; background:rgba(14,165,233,0.04);">
<strong>📑 Contenido de esta guía</strong>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><a href="#tres-herramientas">Tres herramientas distintas</a></li>
<li><a href="#enrutamiento">El agente elige la herramienta correcta</a></li>
<li><a href="#ninguna-herramienta">Cuando ninguna herramienta aplica</a></li>
<li><a href="#cierre">Cierre y próximos pasos</a></li>
</ol>
</div>

<a id="tres-herramientas"></a>

## <span style="color:#F97066;">Tres herramientas distintas</span>

Se definen tres herramientas sin ninguna relación temática entre sí — a propósito, para que la elección del agente dependa claramente del contenido de la pregunta y no de una similitud superficial entre herramientas.

In [1]:
import os
import logging
import math
from dotenv import load_dotenv

load_dotenv()
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def calcular_area_circulo(radio: float) -> str:
    """Calcula el área de un círculo dado su radio.

    Args:
        radio: El radio del círculo.
    """
    area = math.pi * radio ** 2
    return f"{area:.2f}"


@tool
def convertir_moneda(monto: float, tasa_cambio: float) -> str:
    """Convierte un monto de una moneda a otra usando una tasa de cambio dada.

    Args:
        monto: La cantidad a convertir, en la moneda de origen.
        tasa_cambio: Cuántas unidades de la moneda destino equivalen a una unidad de la moneda de origen.
    """
    return f"{monto * tasa_cambio:.2f}"


@tool
def contar_palabras(texto: str) -> str:
    """Cuenta el número de palabras en un texto.

    Args:
        texto: El texto a analizar.
    """
    return str(len(texto.split()))


print("Tres herramientas definidas: calcular_area_circulo, convertir_moneda, contar_palabras.")

Tres herramientas definidas: calcular_area_circulo, convertir_moneda, contar_palabras.


<a id="enrutamiento"></a>

## <span style="color:#F97066;">El agente elige la herramienta correcta</span>

El agente recibe las tres herramientas a la vez. Nada en el código le indica qué herramienta usar para qué pregunta — esa decisión la toma el LLM, en tiempo de ejecución, a partir del nombre, el docstring y los argumentos de cada herramienta.

In [2]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=os.getenv("GOOGLE_API_KEY"))

agente = create_agent(
    model=llm,
    tools=[calcular_area_circulo, convertir_moneda, contar_palabras],
    system_prompt="Usa la herramienta adecuada según la pregunta. Si ninguna aplica, dilo con honestidad.",
)

print("Agente con 3 herramientas construido correctamente.")

Agente con 3 herramientas construido correctamente.


In [3]:
def ejecutar_y_mostrar(consulta):
    resultado = agente.invoke({"messages": [{"role": "user", "content": consulta}]})
    print("Pregunta:", consulta)
    for mensaje in resultado["messages"]:
        if getattr(mensaje, "tool_calls", None):
            for llamada in mensaje.tool_calls:
                print(f"  → Herramienta invocada: {llamada['name']}({llamada['args']})")
    print("Respuesta:", resultado["messages"][-1].text)
    print()


ejecutar_y_mostrar("Convierte 100 dólares a pesos colombianos con una tasa de cambio de 4000.")
ejecutar_y_mostrar("¿Cuántas palabras tiene la frase: el agente eligió bien la herramienta?")
ejecutar_y_mostrar("Calcula el área de un círculo de radio 3.")

Pregunta: Convierte 100 dólares a pesos colombianos con una tasa de cambio de 4000.
  → Herramienta invocada: convertir_moneda({'tasa_cambio': 4000, 'monto': 100})
Respuesta: 100 dólares equivalen a 400,000 pesos colombianos con una tasa de cambio de 4000.



Pregunta: ¿Cuántas palabras tiene la frase: el agente eligió bien la herramienta?
  → Herramienta invocada: contar_palabras({'texto': 'el agente eligió bien la herramienta'})
Respuesta: La frase tiene 6 palabras.



Pregunta: Calcula el área de un círculo de radio 3.
  → Herramienta invocada: calcular_area_circulo({'radio': 3})
Respuesta: El área de un círculo con radio 3 es aproximadamente 28.27.



En los tres casos el agente invocó exactamente la herramienta correspondiente, con los argumentos correctos extraídos de la pregunta en lenguaje natural. Esta elección no está codificada en ningún `if`/`else`: el LLM la infiere de la descripción de cada herramienta, razón por la cual un docstring preciso (como se señaló en <code>1-fundamentos-agentes.ipynb</code>) es tan determinante para el comportamiento del agente.

<a id="ninguna-herramienta"></a>

## <span style="color:#F97066;">Cuando ninguna herramienta aplica</span>

Un agente bien diseñado también debe reconocer cuándo <strong>ninguna</strong> de sus herramientas sirve para responder, en vez de forzar una invocación incorrecta o inventar una respuesta.

In [4]:
ejecutar_y_mostrar("¿Qué hora es en Bogotá ahora mismo?")

Pregunta: ¿Qué hora es en Bogotá ahora mismo?
Respuesta: No tengo acceso a la hora actual ni a herramientas para consultar la hora en tiempo real. Te sugiero revisar un reloj o buscar en internet la hora actual en Bogotá.



Ninguna de las tres herramientas disponibles puede responder esa pregunta (ninguna consulta la hora), y el agente no llamó a ninguna — respondió honestamente que no tiene esa información, en lugar de adivinar una hora o forzar el uso de una herramienta que no corresponde. Esto depende directamente de las instrucciones del <code>system_prompt</code> ("si ninguna aplica, dilo con honestidad"): sin esa instrucción explícita, algunos modelos son más propensos a inventar una respuesta antes que admitir que no pueden resolver la tarea.

<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
Cuantas más herramientas tiene un agente, más difícil es para el LLM elegir correctamente entre ellas — sobre todo si varias tienen propósitos parecidos. En sistemas con muchas herramientas es común agruparlas por especialidad y usar varios agentes (uno por grupo) en lugar de un solo agente con decenas de herramientas; ese patrón se explora en <code>6-multiagentes.ipynb</code>.
</div>

---

<a id="cierre"></a>

# <span style="color:#F97066;">🎯 Cierre y próximos pasos</span>

<div style="border-left:4px solid #14B8A6; background:rgba(20,184,166,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>✅ Resumen</strong><br>
Este notebook mostró cómo un agente elige entre varias herramientas:

- El LLM decide qué herramienta invocar (o ninguna) a partir del nombre, el docstring y los argumentos de cada una — no hay lógica de enrutamiento programada a mano.
- Un docstring claro y específico es lo que permite que esa elección sea correcta.
- Un agente bien instruido reconoce cuándo ninguna herramienta disponible resuelve la pregunta, en vez de forzar una respuesta.

</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>➡️ Continúe con</strong>
<ul style="margin:8px 0 0; padding-left:20px;">
<li><code>4-grafos-langgraph.ipynb</code> — construir el grafo del agente a mano, en vez de depender de la caja negra de <code>create_agent</code>.</li>
</ul>
</div>